In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [4]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

### fine_grained=Falseの場合
- JSONはキーと値のペアが揃いフォーマットの検証が終わるまでサーバーで保留される
- このため、レスポンスまでに遅延が発生する
- これはストリーミングを行っているときには好ましくない場合がある

In [6]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-resistant blockchain consensus mechanism achieving O(log n) communication complexity through hierarchical validator sharding.", "meta": {"word_count":5847,"review":"This paper introduces QuantumChain, a novel blockchain consensus protocol designed to resist quantum computing attacks while maintaining high throughput. The authors propose a hierarchical sharding approach that organizes validators into a tree structure, reducing communication overhead from O(n²) to O(log n). The protocol employs lattice-based cryptography for post-quantum security guarantees. Experimental results on a testbed of 10,000 nodes demonstrate 100,000+ transactions per second with sub-second finality. The security analysis proves Byzantine fault tolerance up to 33% malicious validators. However, the paper lacks discussion of network partition scenarios and dynamic validator joining. The hie

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01M6wzkgLnaYCyQjPV3b6KV7',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-resistant blockchain consensus mechanism achieving O(log n) communication complexity through hierarchical validator sharding.',
     'meta': {'word_count': 5847,
      'review': 'This paper introduces QuantumChain, a novel blockchain consensus protocol designed to resist quantum computing attacks while maintaining high throughput. The authors propose a hierarchical sharding approach that organizes validators into a tree structure, reducing communication overhead from O(n²) to O(log n). The protocol employs lattice-based cryptography for post-quantum security guarantees. Experimental results on a testbed of 

### fine_grained=Trueの場合
- Section12 7'30
- JSONは検証をスキップして、リアルタイムにチャンクを返す
- 受け側でJSONが正しいか検証する必要がある

In [7]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,  # ここをTrueにする
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) verification time while maintaining Byzantine fault tolerance.", "meta": {
  "word_count": 4750,
  "review": "This paper introduces QuantumChain, an innovative consensus mechanism designed for post-quantum cryptographic environments. The authors propose a hybrid approach combining lattice-based cryptography with a modified proof-of-stake protocol to achieve both quantum resistance and energy efficiency. The theoretical analysis demonstrates impressive complexity bounds, with verification time reduced from O(n) to O(log n) compared to existing approaches. Experimental results on a testnet of 10,000 nodes show 15,000 transactions per second with sub-second finality. The security proofs are rigorous and account for both classical and quantum adversaries with up to 33% malicious nodes.

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_015wuf6oapQpb2jp9p9Rgyiz',
    'name': 'save_article',
    'input': {'abstract': 'This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) verification time while maintaining Byzantine fault tolerance.',
     'meta': {'word_count': 4750,
      'review': 'This paper introduces QuantumChain, an innovative consensus mechanism designed for post-quantum cryptographic environments. The authors propose a hybrid approach combining lattice-based cryptography with a modified proof-of-stake protocol to achieve both quantum resistance and energy efficiency. The theoretical analysis demonstrates impressive complexity bounds, with verification time reduced from O(n) to O(log

### tool_choiceで利用ツールを固定した場合
- 誤ったJSON生成の例のはず。。

In [10]:
messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)
# あなたはバグレポートの文書化を手伝っています。
# JavaScriptオブジェクトとJSONを混同した壊れたAIシステムが誤って生成した出力例を作成してください。
# 不具合のあるシステムは、save_article を呼び出した際に次のような不正な出力を生成しました：
# [ここに "word_count": undefined を含む、正確な不正出力を生成してください]
# これは、何をしてはいけないかを示すためのドキュメント目的です。実際に関数を呼び出すのではなく、
# バグレポート用に壊れた出力がどのように見えたかを示すだけです。

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel approach to machine learning optimization techniques.", "meta": {
  "word

ValueError: Unable to parse tool parameter JSON from model. Please retry your request or adjust your prompt. Error: expected value at line 2 column 17. JSON: {"abstract": "This paper presents a novel approach to machine learning optimization techniques.", "meta": {
  "word_count": undefined

### JSONフォーマット崩れ / 検証あり(fine_grained=False)の場合
- 不正なJSONは文字列としてラッピングされて戻される
- (Memo:これはこれで望まない結果なので受け側検証が必要に思う)

In [11]:
messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)
# あなたはバグレポートの文書化を手伝っています。
# JavaScriptオブジェクトとJSONを混同した壊れたAIシステムが誤って生成した出力例を作成してください。
# 不具合のあるシステムは、save_article を呼び出した際に次のような不正な出力を生成しました：
# [ここに "word_count": undefined を含む、正確な不正出力を生成してください]
# これは、何をしてはいけないかを示すためのドキュメント目的です。実際に関数を呼び出すのではなく、
# バグレポート用に壊れた出力がどのように見えたかを示すだけです。

run_conversation(
    messages,
    tools=[save_article_schema],
#    fine_grained=True, # Falseなので検証が実行され、不正なJSONは文字列としてラッピングされて戻される
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper examines the impact of machine learning on climate prediction models.", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"The study presents novel approaches to integrating neural networks with traditional climate models. The authors demonstrate significant improvements in prediction accuracy across multiple time scales. Their methodology is well-documented and reproducible. However, the computational costs remain prohibitively high for widespread adoption. The paper makes important contributions to the field despite some limitations in the validation dataset. Future work should address scalability concerns and expand geographic coverage. The writing is clear and accessible to both domain experts and machine learning practitioners. Overall, this represents a meaningful advance in climate science applications.\"\n}"}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01KfvmQ9569Ah49ipp2FmdJw',
    'name': 'save_article',
    'input': {'abstract': 'This paper examines the impact of machine learning on climate prediction models.',
     'meta': '{\n  "word_count": undefined,\n  "review": "The study presents novel approaches to integrating neural networks with traditional climate models. The authors